In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

def load_sentiments(filepath):
    """从CSV文件加载情感因素列"""
    try:
        df = pd.read_csv(filepath, encoding='utf-8')
        if '情感因素' not in df.columns:
            raise ValueError("CSV文件中未找到'情感因素'列")
        texts = df['情感因素'].dropna().astype(str).tolist()
        print(f"成功加载 {len(texts)} 条情感文本")
        return texts
    except Exception as e:
        print(f"文件加载失败: {str(e)}")
        return []

def safe_clustering(texts, init_threshold=0.77):
    """处理负距离的安全聚类方法"""
    # 1. 使用更大的模型
    model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
    
    # 2. 计算嵌入向量（增加归一化处理）
    print("正在计算文本嵌入...")
    embeddings = model.encode(texts, show_progress_bar=True)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)  # 归一化
    
    # 3. 计算相似度矩阵并确保距离非负
    sim_matrix = cosine_similarity(embeddings)
    dist_matrix = 1 - sim_matrix
    dist_matrix = np.maximum(dist_matrix, 0)  # 确保非负
    np.fill_diagonal(dist_matrix, 0)  # 对角线置零
    
    # 4. 检查距离矩阵有效性
    if np.any(dist_matrix < 0):
        print("警告：距离矩阵包含负值，正在修正...")
        dist_matrix = np.maximum(dist_matrix, 0)
    
    # 5. 层次聚类（增加距离矩阵校验）
    print("正在进行层次聚类...")
    try:
        condensed_dist = squareform(dist_matrix, checks=False)
        Z = linkage(condensed_dist, method='ward')
        
        # 检查连接矩阵中的负距离
        if np.any(Z[:, 2] < 0):
            print("检测到负距离，正在调整...")
            min_nonzero = np.min(Z[Z[:, 2] > 0, 2])
            Z[Z[:, 2] < 0, 2] = min_nonzero * 0.01  # 用最小正距离的1%替换负值
    except Exception as e:
        print(f"层次聚类出错: {str(e)}")
        return None, None, None, None, None
    
    # 6. 寻找最优阈值
    print("\n正在优化聚类阈值...")
    best_score = -1
    best_threshold = init_threshold
    best_clusters = None
    
    thresholds = np.linspace(max(0.6, init_threshold-0.1), 
                           min(0.9, init_threshold+0.1), 
                           15)
    
    for th in thresholds:
        try:
            clusters = fcluster(Z, t=(1-th)*Z[-1, 2]*0.7, criterion='distance')
            
            # 过滤单样本簇
            cluster_counts = pd.Series(clusters).value_counts()
            valid_clusters = cluster_counts[cluster_counts > 1].index
            valid_mask = np.isin(clusters, valid_clusters)
            
            if len(valid_clusters) >= 2:
                score = silhouette_score(dist_matrix[valid_mask][:, valid_mask], 
                                       clusters[valid_mask], 
                                       metric="precomputed")
                print(f"阈值: {th:.3f} | 簇数: {len(cluster_counts)} | 轮廓系数: {score:.4f}")
                
                if score > best_score:
                    best_score = score
                    best_threshold = th
                    best_clusters = clusters
            else:
                print(f"阈值: {th:.3f} | 簇数: {len(cluster_counts)} (无效聚类)")
        except:
            continue
    
    return best_clusters, embeddings, dist_matrix, best_threshold, best_score

def visualize_results(embeddings, clusters, score):
    """可视化聚类结果"""
    pca = PCA(n_components=2)
    points = pca.fit_transform(embeddings)
    
    plt.figure(figsize=(12, 8))
    plt.scatter(points[:,0], points[:,1], c=clusters, cmap='tab20', alpha=0.7)
    plt.colorbar(label='簇ID')
    plt.title(f"文本聚类可视化 (轮廓系数={score:.3f})")
    plt.xlabel("主成分1")
    plt.ylabel("主成分2")
    plt.grid(True)
    plt.show()

if __name__ == "__main__":
    filepath = r"D:\weibo-search\结果文件\南京地铁\情感因素分析\完整分析结果.csv"
    texts = load_sentiments(filepath)
    
    if not texts:
        exit()
    
    # 使用安全聚类方法
    clusters, embeddings, dist_matrix, best_threshold, best_score = safe_clustering(texts)
    
    if clusters is None:
        print("聚类失败，请检查数据或调整参数")
        exit()
    
    # 分析结果
    cluster_counts = pd.Series(clusters).value_counts().sort_index()
    print("\n" + "="*50)
    print(f"最优阈值: {best_threshold:.3f} | 轮廓系数: {best_score:.3f}")
    print(f"总簇数: {len(cluster_counts)}")
    print("簇大小分布:")
    print(cluster_counts)
    
    # 可视化
    visualize_results(embeddings, clusters, best_score)
    
    # 保存结果
    output_df = pd.DataFrame({
        '文本': texts,
        '簇ID': clusters,
        '簇大小': pd.Series(clusters).map(pd.Series(clusters).value_counts())
    })
    output_path = os.path.join(os.path.dirname(filepath), "安全聚类结果1.csv")
    output_df.to_csv(output_path, index=False, encoding='utf_8_sig')
    print(f"\n聚类结果已保存至: {output_path}")